# OIBC 2026 — TSMixer Colab GPU 학습
기존 기상 90/시간 8/정적 146, weighted L1, G1/G2, cutoff·dev 기간을 유지합니다. 기본은 **synthetic 5-step smoke**입니다. 전체 학습은 `MODE="development"`와 `ALLOW_FULL_TRAIN=True`를 직접 설정해야 합니다. 런타임 → 런타임 유형 변경에서 GPU를 선택하세요. T4 등 실제 장치를 자동 확인합니다.

**먼저 이 변경을 GitHub에 반영하세요.** 셀을 위에서 아래로 실행합니다. 재접속 시 같은 RUN_ID로 1번부터 재실행합니다. Drive 세션 메타데이터가 이전 소스 commit을 자동 고정합니다. 세부 설명은 `scripts/tsmixer/colab/README_KO.md`를 참고하세요.

## 1. 실행 모드·경로 설정 — 재접속 시 RUN_ID와 로컬 경로 문자열 유지

In [ ]:
from pathlib import Path
import os, re, sys, json, subprocess, shutil, uuid

REPO_URL = "https://github.com/kimna428/2026_OIBC.git"
REPO_REF = "feature/tsmixer-v0-mac-cuda"  # 첫 실행용; 이후 저장된 commit 자동 사용
REPO_DIR = Path("/content/2026_OIBC")
LOCAL_ROOT = Path("/content/oibc_tsmixer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive/OIBC2026"

MODE = "smoke"                        # "smoke" 또는 "development"
ALLOW_FULL_TRAIN = False              # 전체 학습 시 직접 True로 변경
REAL_DATA_SMOKE = False               # 기본은 synthetic; True이면 실제 이틀 캐시
SMOKE_STEPS = 5                       # 5..10 optimizer updates
RUN_ID = "tsmixer_colab_smoke_v1"       # 전체 학습에는 별도 ID 사용
RESUME = True
LOCAL_QUOTA_GIB = 12                  # 현재 런타임의 개인 작업 공간 예산

assert MODE in ("smoke", "development")
assert MODE != "development" or ALLOW_FULL_TRAIN, "전체 학습은 명시적 허용이 필요합니다"
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", RUN_ID)
assert 5 <= SMOKE_STEPS <= 10
DATA_SOURCE = DRIVE_PROJECT / "data/01_participant_dataset"
DATA_LOCAL = LOCAL_ROOT / "data/01_participant_dataset"
CACHE_ROOT = LOCAL_ROOT / ("cache_real_smoke" if MODE == "smoke" else "cache_full")
RUN_DIR = LOCAL_ROOT / "runs" / RUN_ID
BACKUP_ROOT = DRIVE_PROJECT / "runs" / RUN_ID
NEED_REAL_DATA = MODE == "development" or REAL_DATA_SMOKE
print({"mode": MODE, "run_id": RUN_ID, "local_run": str(RUN_DIR), "drive_backup": str(BACKUP_ROOT)})

## 2. Google Drive Mount — Google 계정 승인 필요

In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
assert (DRIVE_MOUNT / "MyDrive").is_dir(), "Drive mount를 확인하세요"
SESSION_REF = BACKUP_ROOT / "session_ref.json"
PINNED_REF = REPO_REF
if SESSION_REF.exists():
    assert RESUME, "기존 Drive 실험입니다. Resume하거나 새로운 RUN_ID를 사용하세요"
    prior = json.loads(SESSION_REF.read_text())
    assert prior["mode"] == MODE, "smoke checkpoint를 전체 학습으로 승격할 수 없습니다"
    assert re.fullmatch(r"[0-9a-f]{40,64}", prior["git_commit"])
    PINNED_REF = prior["git_commit"]
print("Checkout ref:", PINNED_REF)

## 3. GitHub Clone 및 소스 commit 고정 — 기존 checkout을 reset/clean하지 않음

In [ ]:
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
assert (REPO_DIR / ".git").exists()
assert subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip() == REPO_URL
assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip(), "수정된 checkout은 보존하고 새 경로를 선택하세요"
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", PINNED_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert (REPO_DIR / "analysis/tsmixer/persistence.py").is_file(), "Colab 변경을 먼저 commit/push하세요"
print("Git commit:", COMMIT)

## 4. 의존성 설치 — Colab의 CUDA PyTorch 유지
공통 패키지만 설치하며 torch를 CPU wheel로 덮어쓰지 않습니다. 필수 import가 실패하면 환경을 해결한 뒤 진행하세요. 전체 pip 환경의 충돌 기록도 보관합니다.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "analysis/tsmixer/requirements-common.txt"], check=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(LOCAL_ROOT / "pip_check.txt").write_text(pip_check.stdout + pip_check.stderr)
print(pip_check.stdout or "pip check: no output", "exit:", pip_check.returncode)
import torch, numpy, pandas, pyarrow, yaml
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 5)
assert int(torch.__version__.split(".")[0]) < 3
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(LOCAL_ROOT / "environment_freeze.txt").write_text(freeze)
print("Python", sys.version, "torch", torch.__version__, "numpy", numpy.__version__)

## 5. GPU·디스크·기존 디바이스 호환성 확인
전체 학습은 Linux CUDA가 필수입니다. 기본 fp32이며 T4에서 bf16을 강제하지 않습니다. GPU·버전이 다른 재개는 bitwise 동일성을 보장하지 않습니다.

In [ ]:
from analysis.tsmixer.runtime import device_for, environment
DEVICE = device_for("auto")
if MODE == "development" or REAL_DATA_SMOKE:
    assert DEVICE.type == "cuda", "GPU 런타임을 선택하세요"
print(json.dumps(environment(DEVICE), indent=2))
if DEVICE.type == "cuda":
    subprocess.run(["nvidia-smi"], check=True)
print("Local free GiB:", shutil.disk_usage(LOCAL_ROOT).free / 1024**3)
os.environ.update(OIBC_DATA_ROOT=str(DATA_LOCAL), OIBC_TSMIXER_CACHE_ROOT=str(CACHE_ROOT),
                  OIBC_TSMIXER_RUN_ROOT=str(LOCAL_ROOT / "runs"), OIBC_STORAGE_ROOT=str(LOCAL_ROOT),
                  OIBC_QUOTA_GIB=str(LOCAL_QUOTA_GIB), PYTHONDONTWRITEBYTECODE="1")

## 6. 학습 설정 고정 및 Drive 실험 ID 연결
모델·loss·split은 그대로이고 Scheduler는 원래와 같은 constant LR입니다. smoke 설정의 epoch 값은2이며 실제 smoke 종료는5~10 optimizer step 제한을 사용합니다. 같은 ID에서 설정을 바꾸면 거부합니다.

In [ ]:
from analysis.tsmixer.config import load_config, validate
from analysis.tsmixer.colab import bind_session, stage_dataset, run_logged
from analysis.tsmixer.persistence import restore_latest, backup_run, valid_checkpoint
CONFIG = load_config("configs/tsmixer/colab.yaml")
if MODE == "smoke":
    CONFIG["training"]["epochs"] = 2
validate(CONFIG)
assert CONFIG["split"] == {"outer_fold": 5, "dev_start": "2025-02-01", "dev_end": "2025-02-28"}
assert CONFIG["training"]["seed"] == 2026
bind_session(BACKUP_ROOT, COMMIT, CONFIG, MODE)
LOCAL_CONFIG = LOCAL_ROOT / (RUN_ID + ".yaml")
LOCAL_CONFIG.write_text(yaml.safe_dump(CONFIG, sort_keys=False))
print(LOCAL_CONFIG.read_text())

## 7. Drive 원본 → Colab 로컬 디스크 복사
기본 synthetic smoke에서는 건너뜁니다. 실제 데이터는 필수8개만 스트리밍 복사하고 제공 inventory의 SHA-256을 검사합니다. 기존 로컬 파일이 다르면 덮어쓰지 않습니다.

In [ ]:
if NEED_REAL_DATA:
    copied = stage_dataset(DATA_SOURCE, DATA_LOCAL, REPO_DIR / "reports/tsmixer/data_inventory.json")
    print("Local data verified:", len(copied), "files")
else:
    print("Synthetic smoke: 원본 복사와 대용량 캐시 생성 생략")

## 8. 로컬 compact 캐시 확인·생성 — 기존 전처리 재사용
전체 학습은 원본 전체 시간축, real smoke는 2025-01-30~01-31만 준비합니다. 원본은 Drive에서 직접 학습하지 않습니다. 실패한 부분 캐시는 지우거나 덮어쓰지 않습니다.

In [ ]:
if NEED_REAL_DATA:
    if (CACHE_ROOT / "train/manifest.json").is_file():
        subprocess.run([sys.executable, "-m", "analysis.tsmixer", "check-data", "--config", str(LOCAL_CONFIG),
                        "--cache-root", str(CACHE_ROOT), "--checksums", "--reference", "reports/tsmixer/data_inventory.json"], check=True)
    else:
        assert not CACHE_ROOT.exists() or not any(CACHE_ROOT.iterdir()), "부분 캐시는 보존하고 새 런타임에서 재생성하세요"
        if MODE == "smoke":
            args = [sys.executable, "-m", "analysis.deep_learning.prepare", "--data-root", str(DATA_LOCAL),
                    "--cache-root", str(CACHE_ROOT), "--prepared-root", str(LOCAL_ROOT / "unused_prepared"),
                    "--partition", "train", "--layout", "compact", "--start", "2025-01-30", "--end", "2025-02-01"]
        else:
            args = [sys.executable, "-m", "analysis.tsmixer", "prepare-cache", "--config", str(LOCAL_CONFIG),
                    "--device", "cuda", "--checksums", "--reference", "reports/tsmixer/data_inventory.json"]
        subprocess.run(args, check=True)
else:
    print("Synthetic cache는 smoke CLI가 작은 fixture로 생성합니다")

## 9. 마지막 정상 checkpoint 복원
COMMITTED marker·파일 크기·SHA-256·checkpoint 상태를 확인합니다. 최신 백업이 불완전하거나 손상되면 이전 정상 세대로 돌아갑니다. 모든 백업이 손상되면 새 학습으로 조용히 시작하지 않습니다.

In [ ]:
if not RUN_DIR.exists() and RESUME:
    selected = restore_latest(BACKUP_ROOT, RUN_DIR)
    print("Restored:", selected or "이전 백업 없음; 새 실험")
if RUN_DIR.exists():
    assert RESUME and (RUN_DIR / "last.pt").is_file(), "기존 디렉터리는 덮어쓰지 않습니다"
    CHECKPOINT = valid_checkpoint(RUN_DIR / "last.pt")
    assert CHECKPOINT["smoke_test"] == (MODE == "smoke")
    RESUME_FILE = RUN_DIR / "last.pt"
    DONE = (CHECKPOINT["global_step"] >= SMOKE_STEPS if MODE == "smoke" else
            CHECKPOINT["epoch"] >= CONFIG["training"]["epochs"] or
            CHECKPOINT["bad_epochs"] >= CONFIG["training"]["patience"])
    print("Saved epoch", CHECKPOINT["epoch"], "step", CHECKPOINT["global_step"], "complete", DONE)
else:
    RESUME_FILE, DONE = None, False

## 10. 학습 또는 5-step Smoke — 매 완료 epoch Drive 백업
동기 백업 실패 시 학습을 멈추며 이전 정상 백업은 보존합니다. 끊긴 epoch는 마지막 완료 epoch부터 다시 계산합니다. 강제 런타임 종료 직전의 미완료 작업까지 보존되지는 않습니다.

In [ ]:
CONSOLE = LOCAL_ROOT / "logs" / (RUN_ID + ".log")
if RESUME_FILE and (RUN_DIR / "console.log").is_file() and not CONSOLE.exists():
    CONSOLE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(RUN_DIR / "console.log", CONSOLE)
if not DONE:
    args = [sys.executable, "-m", "analysis.tsmixer", "smoke" if MODE == "smoke" else "train",
            "--config", str(LOCAL_CONFIG), "--device", "auto", "--output", str(RUN_DIR),
            "--backup-root", str(BACKUP_ROOT)]
    if MODE == "smoke":
        args += ["--steps", str(SMOKE_STEPS)]
        if REAL_DATA_SMOKE: args += ["--real-cache"]
    else:
        args += ["--full-train"]
    if RESUME_FILE: args += ["--resume", str(RESUME_FILE)]
    try:
        run_logged(args, CONSOLE)
    finally:
        if (RUN_DIR / "last.pt").is_file():
            shutil.copyfile(LOCAL_CONFIG, RUN_DIR / "config.yaml")
            shutil.copyfile(LOCAL_ROOT / "environment_freeze.txt", RUN_DIR / "environment_freeze.txt")
            if CONSOLE.is_file(): shutil.copyfile(CONSOLE, RUN_DIR / "console.log")
            print("Final backup:", backup_run(RUN_DIR, BACKUP_ROOT))
else:
    print("이미 완료한 학습입니다. validation/백업 셀로 진행하세요")

## 11. 검증 및 최종 development 예측 CSV
전체 학습에서는 기존 dev 평가만 실행합니다. 출력 CSV는 선택된 dev 예측이며 독립 OOF나 공식 test 제출물이 아닙니다. Smoke에서는 작은 synthetic 평가 정합성 테스트만 실행합니다.

In [ ]:
if MODE == "smoke":
    subprocess.run([sys.executable, "-m", "unittest",
                    "analysis.tsmixer.tests.test_tsmixer.DataTests.test_streaming_evaluation_matches_manual_canonical_predictions", "-v"], check=True)
    print("Smoke의 loss/checkpoint reload 결과:", (RUN_DIR / "summary.json").read_text() if (RUN_DIR / "summary.json").is_file() else "checkpoint에서 재개된 완료 run")
else:
    EVAL_DIR = RUN_DIR / "eval_dev"
    from analysis.tsmixer.persistence import sha256
    expected_checkpoint_hash = sha256(RUN_DIR / "best.pt")
    reusable = False
    if (EVAL_DIR / "metrics.json").is_file():
        prior_metrics = json.loads((EVAL_DIR / "metrics.json").read_text())
        reusable = prior_metrics.get("checkpoint_sha256") == expected_checkpoint_hash
    if EVAL_DIR.exists() and not reusable:
        EVAL_DIR.rename(RUN_DIR / ("interrupted_or_stale_eval_" + uuid.uuid4().hex[:8]))
    if not (EVAL_DIR / "metrics.json").is_file():
        try:
            run_logged([sys.executable, "-m", "analysis.tsmixer", "evaluate", "--config", str(LOCAL_CONFIG),
                        "--device", "auto", "--checkpoint", str(RUN_DIR / "best.pt"),
                        "--cache-root", str(CACHE_ROOT), "--output", str(EVAL_DIR), "--export-csv"],
                       RUN_DIR / "evaluation_console.log")
        finally:
            print("Evaluation backup:", backup_run(RUN_DIR, BACKUP_ROOT))
    else:
        metrics = json.loads((EVAL_DIR / "metrics.json").read_text())
        assert metrics["split"] == "dev" and metrics["holdout_used"] is False
        if not (EVAL_DIR / "csv").is_dir():
            from analysis.tsmixer.colab import export_development
            from analysis.tsmixer.data import open_cache
            from analysis.tsmixer.checkpoint import load
            export_development(EVAL_DIR, open_cache(CACHE_ROOT / "train"), load(RUN_DIR / "best.pt")["plan"])
    print((EVAL_DIR / "metrics.json").read_text())

## 12. 최종 Drive 백업·산출물 확인
파일 검증을 마친 generation만 복원 후보입니다. Drive/FUSE의 원격 영속성은 서비스의 보장 범위에 따릅니다. 원본·캐시는 백업하지 않고 재접속 시 로컬에서 다시 준비합니다.

In [ ]:
last_generation = backup_run(RUN_DIR, BACKUP_ROOT)
print("Verified Drive generation:", last_generation)
print("Local outputs:")
for file in sorted(RUN_DIR.rglob("*")):
    if file.is_file() and "synthetic_cache" not in file.parts:
        print(file.relative_to(RUN_DIR), file.stat().st_size, "bytes")
print("재접속: 같은 RUN_ID·MODE로 1~12번을 다시 실행하세요. session_ref.json의 commit이 자동 선택됩니다.")